# SiteSight M2: caption fields, embeddings, near-duplicate groups, leakage-free split

Runs the same command-line entry points as the laptop (`make fields`, `make embed`, `make curate`),
so there is no logic in this notebook. **Runtime > Change runtime type > T4 GPU** speeds up the
embedding step (CPU also works, about 4 minutes).

Before the first run: add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar, turn on
*Notebook access*). It must be a fine-grained GitHub token limited to `hddaghigh/sitesight` with
**Contents: Read-only**. The token is never printed or stored in the repo.

In [ ]:
import base64, os, subprocess
from google.colab import userdata

REPO = "https://github.com/hddaghigh/sitesight.git"
DEST = "/content/sitesight"
token = userdata.get("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
header = f"http.extraheader=AUTHORIZATION: basic {auth}"  # keeps the token out of the remote URL
cmd = (
    ["git", "-C", DEST, "-c", header, "pull", "--ff-only"]
    if os.path.isdir(DEST)
    else ["git", "-c", header, "clone", "--depth", "1", REPO, DEST]
)
r = subprocess.run(cmd, capture_output=True, text=True)
print(
    "repo ready"
    if r.returncode == 0
    else "git failed: " + (r.stderr or r.stdout).replace(token, "***")
)
print(
    subprocess.run(
        ["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True
    ).stdout
)

In [ ]:
%cd /content/sitesight
%pip -q install -e . transformers
import torch

print(
    "torch",
    torch.__version__,
    "| GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)",
)

In [ ]:
DATA = "/content/sitesight_data"
# Downloads 566 MiB, verifies SHA-256, extracts. Idempotent.
!python scripts/download_data.py --data-dir {DATA} --min-free-gb 3

In [ ]:
!python -m sitesight.data.via_parse --via {DATA}/extracted/via_region_data_final.json \
    --images {DATA}/extracted/UAVdata --out {DATA}/derived/annotations.json --report {DATA}/derived/validation_report.json
!python -m sitesight.data.caption_fields --annotations {DATA}/derived/annotations.json \
    --out {DATA}/derived/annotations_fields.json --report {DATA}/derived/coverage_report.json

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
!python scripts/embed_images.py --annotations {DATA}/derived/annotations_fields.json \
    --images {DATA}/extracted/UAVdata --out {DATA}/derived/dinov2_small.npy --device {device}

In [ ]:
!python -m sitesight.data.curate --config configs/data.toml --annotations {DATA}/derived/annotations_fields.json \
    --images {DATA}/extracted/UAVdata --embeddings {DATA}/derived/dinov2_small.npy --out-dir {DATA}/derived/curated

## Compare with the committed reference run (laptop CPU)

GPU and CPU floating-point results can differ in the last digits, so a similarity right at a threshold
may flip an edge. Large differences would mean something is wrong.

In [ ]:
import json

new = json.load(open(f"{DATA}/derived/curated/curation_report.json"))
ref = json.load(open("reports/m2/curation_report.json"))
for key in ("n_near_duplicate_pairs",):
    print(key, "colab:", new[key], "reference:", ref[key])
print("groups   colab:", new["groups"]["n_groups"], "reference:", ref["groups"]["n_groups"])
print("splits   colab:", new["splits"]["images"], "reference:", ref["splits"]["images"])
print(
    "naive leak mean colab:",
    round(new["naive_split_leakage"]["train_test_pairs_mean"], 1),
    "reference:",
    round(ref["naive_split_leakage"]["train_test_pairs_mean"], 1),
)
print("grouped leak colab:", new["grouped_split_leakage"]["crossing_train_test"])

In [ ]:
# Save small outputs only: JSON reports, splits, curated annotations and the 2 MB embedding matrix.
!cd {DATA}/derived && zip -qr /content/m2_outputs.zip validation_report.json coverage_report.json dinov2_small.npy curated/*.json
from google.colab import files

files.download("/content/m2_outputs.zip")